In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access the variables using os.environ or os.getenv
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print(GOOGLE_API_KEY)

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_CSE_ID"] = os.getenv("GOOGLE_CSE_ID")

In [ ]:
from pathlib import Path

file_name = "KapilDev_C++Dev_8_Resume.pdf"
file_path = Path.cwd() / file_name

print(file_path)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path)
docs = loader.load()

if docs is None:
    print("No documents found")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=50, # 10% overlap to maintain context between chunks
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(docs)

if chunks is None:
    print("No chunks found")
else:
    for index, chunk in enumerate(chunks):
        print(f"\n************ Chunk {index+1} ************\n")
        print(chunk.page_content)
        print('-'*100)


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

google_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents = chunks, embedding=google_embeddings, persist_directory="./chroma_db")

if vectorstore is None:
    print("Vector store is null")


retriever = vectorstore.as_retriever()

### Third Try

In [ ]:
# Assuming 'vectorstore' is your existing Chroma/FAISS/DocArray instance
# search_kwargs={"k": 3} tells it to fetch the top 3 most relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Senior Tip: To turn this into a Tool for an agent, wrap it like this:
# from langchain.tools.retriever import create_retriever_tool
from langchain_core.tools import tool, create_retriever_tool

rag_tool = create_retriever_tool(
    retriever,
    name="pdf_search",
    description="Searches and returns information from the uploaded PDF document. Use this as your first source."
)

In [ ]:

# 1. Imports for the new standard
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_community import GoogleSearchAPIWrapper
from langchain_core.tools import Tool
from langgraph.types import RetryPolicy

# Note: You still use the 'rag_tool' and 'search_tool' we defined earlier

# 2. Initialize Gemini 2.5 Flash Lite
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# llm_with_retry = llm.with_retry(
#     stop_after_attempt=5,
#     wait_exponential_jitter=True
# )


# Properly initialize the search wrapper
# Ensure GOOGLE_CSE_ID and GOOGLE_API_KEY are in your environment
search_wrapper = GoogleSearchAPIWrapper()

google_search_tool = Tool(
    name="google_search",
    description="Search the web for current events and general knowledge.",
    func=search_wrapper.run,
)

# Use the list of tool objects
tools = [rag_tool, google_search_tool]



# 4. Create the Agent
# This replaces create_tool_calling_agent + AgentExecutor
agent_executor = create_react_agent(llm, tools)

In [ ]:
# Prepare the input in the new message format
inputs = {"messages": [("user", "Summarize KAPIL DEV MUTHU experience 3 sentences. Specify all his roles. Format the output neatly")]}

# This will wait for all tool calls to finish before returning
result = agent_executor.invoke(inputs, config={"recursion_limit": 5})

# 3. Print the final message from the assistant
# The 'messages' list contains the full history; [-1] is the final response
final_answer = result["messages"][-1].content
print("Assistant:", final_answer)
